# 4. LLM-режим команды Graph-EVT-agent

**Цель:** запустить детерминированный численный `GraphEVTPipeline`, а затем передать его ограниченный evidence-пакет специализированным LLM-агентам из Graph-EVT-agent. LLM объясняет и проверяет результат, но не меняет порог, граф или ranking.

Для живого запуска задайте `MISTRAL_API_KEY` в окружении или введите его в скрытом prompt ячейки ниже. Ключ не записывается в ноутбук или артефакты.

In [1]:
from pathlib import Path
import os
from getpass import getpass
from urllib.error import HTTPError, URLError
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig
from graph_evt_agent.agents import EVTAgentTeam, MistralClient

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/generated/fractal_extreme_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1


In [3]:
api_key = os.getenv("MISTRAL_API_KEY") or getpass("Mistral API key (input hidden): ").strip()
MISTRAL_READY = False
mistral_client = None

if api_key:
    try:
        mistral_client = MistralClient(api_key=api_key)
        check_reply = mistral_client.complete(
            "Reply exactly with MISTRAL_OK.",
            "Connection check. Reply exactly with MISTRAL_OK.",
        )
        MISTRAL_READY = bool(check_reply.strip())
        print("Mistral API check passed." if MISTRAL_READY else "Mistral returned an empty response.")
    except HTTPError as error:
        print(f"Mistral API check failed with HTTP {error.code}; verify that the key is active and has API access.")
    except URLError as error:
        print(f"Mistral API check could not reach the service: {error.reason}")
    except Exception as error:
        print(f"Mistral API check failed ({type(error).__name__}).")
else:
    print("Mistral API check skipped: no key was provided.")


Mistral API check could not reach the service: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)


In [5]:
BASELINE_SIZE = int(0.75 * len(x))
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.95, alarm_probability=0.999, persistence=3),
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
)
# Численный результат доступен и воспроизводим независимо от LLM.
numeric_result = pipeline.run_detailed(x)
{"detected": numeric_result.detection.detected, "time_index": numeric_result.detection.time_index}

{'detected': False, 'time_index': None}

In [6]:
if MISTRAL_READY:
    team = EVTAgentTeam(pipeline, mistral_client)
    try:
        agent_report = team.run(
            x,
            task="Проверь EVT-детекцию и ограничения вывода для одномерного синтетического ряда",
        )
        print(agent_report.final_report)
    except Exception as error:
        print(f"LLM review failed after the preflight check ({type(error).__name__}).")
else:
    print("LLM review skipped; the numeric pipeline result is still available.")


LLM review skipped; the numeric pipeline result is still available.


`EVTAgentTeam` сначала вызывает тот же детерминированный pipeline. Агентам передаются агрегированные evidence-поля, а не право переписать вычисления. Для 1-D входа графовый и локализационный агенты корректно пропускаются. Текст LLM является интерпретацией и должен проверяться по `numeric_result`.